# 🏗️ Baseline Inference & SAM Exploration

This notebook demonstrates:
1. **Baseline inference** using pre-trained YOLOv8 on COCO
2. **Fine-tuned model inference** on construction site images
3. **SAM (Segment Anything Model) exploration** — what worked and what didn't

> **No Roboflow account or API key required.** The dataset is stored directly in the repository.

---

## 1. Setup

In [ ]:
!pip install ultralytics==8.2.0 -q

from ultralytics import YOLO, SAM
from IPython.display import Image, display
import torch
import os
import glob

print(f"GPU available: {torch.cuda.is_available()}")

In [ ]:
# ============================================================
# Clone the repository — dataset is inside 'images dataset/'
# IMPORTANT: Replace with YOUR GitHub repo URL
# ============================================================
REPO_URL = "https://github.com/musleh557790473-cell/Construction-Site-Safety-Monitoring-YOLOv8-Object-Detection.git"
REPO_NAME = "Construction-Site-Safety-Monitoring-YOLOv8-Object-Detection"

if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}
    print("✅ Repository cloned")
else:
    print("✅ Repository already exists")

DATASET_DIR = os.path.join(REPO_NAME, "images dataset")
SAMPLE_IMAGES = os.path.join(DATASET_DIR, "valid", "images")
print(f"Sample images dir: {SAMPLE_IMAGES}")
print(f"Images found: {len(glob.glob(os.path.join(SAMPLE_IMAGES, '*')))}")

## 2. Baseline Inference (Pre-trained COCO model)

First, let's see how a generic YOLOv8 model (trained on COCO) performs on construction site images. This establishes our **baseline** — what we get with zero domain-specific training.

In [ ]:
# Load pre-trained COCO model
baseline_model = YOLO("yolov8n.pt")

print("COCO classes relevant to construction:")
relevant_classes = {0: 'person', 2: 'car', 5: 'bus', 7: 'truck'}
for idx, name in relevant_classes.items():
    print(f"  Class {idx}: {name}")
print("\n⚠️ Note: COCO has no PPE classes (helmet, vest, etc.)")
print("   This is why domain-specific training is needed.")

In [ ]:
# Run baseline predictions on sample images from the dataset
sample_imgs = sorted(glob.glob(os.path.join(SAMPLE_IMAGES, '*')))[:5]

if sample_imgs:
    baseline_results = baseline_model.predict(
        source=sample_imgs,
        conf=0.25,
        save=True,
        project="baseline",
        name="coco-preds",
        exist_ok=True
    )
    
    pred_images = sorted(glob.glob("baseline/coco-preds/*.jpg") + 
                        glob.glob("baseline/coco-preds/*.png"))
    for img in pred_images:
        print(f"\n🖼️ Baseline: {os.path.basename(img)}")
        display(Image(filename=img, width=600))
else:
    print("⚠️ No images found in dataset. Check the clone step.")

### Baseline Observations

The COCO-pretrained model:
- ✅ Detects `person` (workers) reasonably well
- ✅ Detects `truck` and `car` vehicles on site
- ❌ Cannot detect PPE items (helmets, vests) — not in COCO classes
- ❌ Cannot distinguish PPE compliance (worn vs. not worn)
- ❌ Does not detect construction-specific objects (barriers, machinery)

**Conclusion:** Domain-specific training is essential for construction safety monitoring.

## 3. Fine-tuned Model Inference

In [ ]:
# ============================================================
# Load your fine-tuned weights
# Option 1: From training run in this session
# Option 2: Upload best.pt from GitHub Releases
# ============================================================

FINETUNED_WEIGHTS = "best.pt"  # Update path as needed

if os.path.exists(FINETUNED_WEIGHTS):
    finetuned_model = YOLO(FINETUNED_WEIGHTS)
    print(f"✅ Loaded fine-tuned model: {FINETUNED_WEIGHTS}")
    print(f"   Classes: {finetuned_model.names}")
else:
    print("⚠️ Fine-tuned weights not found.")
    print("   Upload best.pt or run the training notebook first.")

In [ ]:
# Compare: Fine-tuned model on the same images
if os.path.exists(FINETUNED_WEIGHTS) and sample_imgs:
    ft_results = finetuned_model.predict(
        source=sample_imgs,
        conf=0.25,
        save=True,
        project="finetuned",
        name="construction-preds",
        exist_ok=True
    )
    
    ft_images = sorted(glob.glob("finetuned/construction-preds/*.jpg") + 
                      glob.glob("finetuned/construction-preds/*.png"))
    for img in ft_images:
        print(f"\n🖼️ Fine-tuned: {os.path.basename(img)}")
        display(Image(filename=img, width=600))

## 4. SAM (Segment Anything Model) Exploration

We explored Meta's Segment Anything Model (SAM) as a complementary tool for construction site analysis.

In [ ]:
# ============================================================
# SAM Exploration
# Note: SAM requires significant GPU memory
# ============================================================

if torch.cuda.is_available():
    try:
        sam_model = SAM("sam_b.pt")  # SAM base model
        print("✅ SAM model loaded")
        
        # Run SAM on a sample image
        if sample_imgs:
            sam_results = sam_model.predict(
                source=sample_imgs[0],
                save=True,
                project="sam-exploration",
                name="segments",
                exist_ok=True
            )
            print(f"SAM segments found: {len(sam_results[0].masks) if sam_results[0].masks else 0}")
            
            # Display result
            sam_imgs = glob.glob("sam-exploration/segments/*")
            if sam_imgs:
                display(Image(filename=sam_imgs[0], width=600))
    except Exception as e:
        print(f"SAM exploration failed: {e}")
        print("This is expected if GPU memory is limited.")
else:
    print("⚠️ SAM requires GPU. Skipping exploration.")
    print("   See notes below for SAM findings from previous runs.")

### SAM Exploration Notes

#### What Helped ✅
- **Precise segmentation masks** — SAM provided pixel-level segmentation of workers and equipment, which could complement YOLOv8's bounding boxes
- **Zero-shot capability** — SAM segments objects without needing construction-specific training
- **Point/box prompting** — Using YOLOv8 bounding boxes as SAM prompts produced high-quality segmentation masks

#### What Failed ❌
- **Speed** — SAM is too slow for real-time safety monitoring (~2-5 sec per image vs. YOLOv8's ~20ms)
- **No class labels** — SAM segments everything but cannot distinguish between helmet vs. no-helmet
- **GPU memory** — SAM ViT-H requires >8GB VRAM; not feasible on Colab free tier consistently
- **Over-segmentation** — In cluttered construction scenes, SAM produces too many irrelevant segments

#### Conclusion
SAM is useful as a **secondary tool** for generating training data or refining annotations, but **not suitable as the primary detection model** for real-time construction safety monitoring. YOLOv8 remains the better choice for this application.

## 5. Side-by-Side Comparison Summary

| Feature | COCO Baseline | Fine-tuned YOLOv8 | SAM |
|---------|:------------:|:-----------------:|:---:|
| Detects workers | ✅ | ✅ | ✅ (no label) |
| Detects PPE | ❌ | ✅ | ❌ |
| PPE compliance | ❌ | ✅ | ❌ |
| Detects machinery | ❌ | ✅ | ✅ (no label) |
| Detects barriers | ❌ | ✅ | ✅ (no label) |
| Real-time speed | ✅ | ✅ | ❌ |
| Pixel-level masks | ❌ | ❌ | ✅ |

**Winner for this use case:** Fine-tuned YOLOv8